# textseg: train the text-map detector (GPU)

The project's own detector (`ml/textseg/`): a timm MobileNetV3 backbone
(ImageNet, Apache-2.0), an FPN and one text-probability map, trained with a
DBNet-style loss and an ignore mask.

**Settings:** Accelerator GPU (T4 or P100), Internet on. **Inputs:** the code
dataset (`ml/textseg/pack_kaggle.py`) and the `textseg-cache` notebook's
output. Then Save Version -> Save & Run All.

One version trains every seed in `SEEDS`. Each lands under `seed<N>/`: `run/`
(checkpoints, `results.csv`), `textseg.onnx`, `score_*.txt` and
`sweep_textseg.json`. Compare recipes over several seeds; `val_seed` stays
fixed so every seed validates on the same pages. The exported weights are the
EMA at its best validation epoch (`best.pt`).


In [ ]:
CONFIG = dict(
    epochs=40,
    batch=16,
    crop=768,
    lr=1e-3,
    backbone='mobilenetv3_large_100',
    val_seed=0,        # fixed across seeds: same val pages for every run
    ema=0.999,
    max_hours=2.5,     # per seed; 3 seeds fit a 9 h session
)
SEEDS = [0, 1, 2]
RUN_TRAINING = True
# To continue a stopped version: attach its output and set this to its root
# (the directory holding seed<N>/). Finished seeds are skipped; a seed with
# run/last.pt resumes.
RESUME_FROM = None

In [ ]:
import glob, shutil, zipfile
from pathlib import Path

# The code dataset; unpack the zip if Kaggle kept it whole.
WORK = Path('/kaggle/working/comic-translator')
if not (WORK / 'ml/textseg/train.py').exists():
    found = glob.glob('/kaggle/input/**/ml/textseg/train.py', recursive=True)
    if found:
        shutil.copytree(Path(found[0]).parents[2], WORK, dirs_exist_ok=True)
    else:
        archive = glob.glob('/kaggle/input/**/comic-translator-ml.zip', recursive=True)
        assert archive, 'attach the comic-translator-ml dataset (Add Input)'
        zipfile.ZipFile(archive[0]).extractall(WORK)
assert (WORK / 'ml/textseg/train.py').exists()
print('code at', WORK)

In [ ]:
!pip install -q timm onnx onnxruntime 2>&1 | tail -1
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'set Accelerator to a GPU in the notebook settings'

In [ ]:
labels = [p for p in glob.glob('/kaggle/input/**/labels.json', recursive=True) if Path(p).with_name('pages').is_dir()]
assert labels, 'attach the textseg-cache notebook output (Add Input -> Your Work)'
CACHE = Path(labels[0]).parent
OUT = Path('/kaggle/working')
if RESUME_FROM:
    for previous in Path(RESUME_FROM).glob('seed*'):
        shutil.copytree(previous, OUT / previous.name, dirs_exist_ok=True)
print('cache', CACHE, len(list((CACHE / 'pages').glob('*.jpg'))), 'pages')

In [ ]:
import os
args = ' '.join(f'--{k.replace("_", "-")} {v}' for k, v in CONFIG.items())
for seed in SEEDS:
    SEED_DIR = OUT / f'seed{seed}'
    RUN = SEED_DIR / 'run'
    ONNX = SEED_DIR / 'textseg.onnx'
    if ONNX.exists():
        print(f'seed {seed}: {ONNX} already exported, skipping')
        continue
    print(f'==== seed {seed} ====', flush=True)
    if RUN_TRAINING:
        !cd {WORK} && python ml/textseg/train.py --cache {CACHE} --out {RUN} --workers {os.cpu_count()} --seed {seed} {args}
    !cd {WORK} && python ml/textseg/export.py --checkpoint {RUN}/best.pt --out {ONNX}
    for th in ('0.25', '0.50'):
        !cd {WORK} && python ml/eval/score_detect.py --model {ONNX} --threshold {th} | tee {SEED_DIR}/score_{th}.txt
    !cd {WORK} && python ml/scripts/sweep_detect.py --model textseg_seed{seed}={ONNX} --out {SEED_DIR}/sweep_textseg.json | tail -25

In [ ]:
# Each seed trains, exports and scores in the cell above, so a failure keeps
# earlier seeds' outputs.

## Scoring

Done per seed above with `score_detect.py` and `sweep_detect.py`. Merge the
seeds' `sweep_textseg.json` files locally with `sweep_detect.py --load`.


In [ ]:
!ls -la {OUT} {OUT}/seed*

In [ ]:
# Keep the output small: pages are inputs, not results.
shutil.rmtree(WORK / 'ml/eval/data', ignore_errors=True)